- **Name:** 20.5_streaming_aggregations
- **Author:** Shamas Imran
- **Desciption:** Performing aggregations in structured streaming
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Applied groupBy aggregations on streams  
                                              Used tumbling and sliding windows  
                                              Demonstrated append vs update modes  
-->

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import col, window, session_window

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 18, Finished, Available, Finished)

In [17]:
# ------------------------------------------------------------
# 1) Spark Session
# ------------------------------------------------------------
spark = (
    SparkSession.builder
        .appName("Aggregations_and_WindowedAggregations")
        .getOrCreate()
)

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 19, Finished, Available, Finished)

In [18]:
rootPath = "abfss://shamas_ws@onelake.dfs.fabric.microsoft.com/test_Lakehouse.Lakehouse/Files/"
masterPath = rootPath + "spark-streaming/"
inputPath       = masterPath + "csv_input"
checkpointPath  = masterPath + "checkpoints/aggregations"
outputPath      = masterPath + "csv_output"

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 20, Finished, Available, Finished)

In [19]:
# Check if folder exists before deleting
if mssparkutils.fs.exists(masterPath):
    mssparkutils.fs.rm(masterPath, True)  # True = recursive delete

# Create directories
mssparkutils.fs.mkdirs(masterPath)
mssparkutils.fs.mkdirs(inputPath)
mssparkutils.fs.mkdirs(checkpointPath)
mssparkutils.fs.mkdirs(outputPath)

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 21, Finished, Available, Finished)

True

In [20]:
# ------------------------------------------------------------
# 3) Define Schema
# ------------------------------------------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("event_time", TimestampType(), True)
])

# ------------------------------------------------------------
# 4) Create Streaming DataFrame
# ------------------------------------------------------------
df_stream = (
    spark.readStream
         .option("header", "true")
         .schema(schema)
         .csv(inputPath)
)

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 22, Finished, Available, Finished)

In [27]:
# ------------------------------------------------------------
# 6) Aggregation WITH Watermark
# ------------------------------------------------------------
# Keep state for 5 minutes; late events beyond 5 min are dropped
from pyspark.sql.functions import window

# Add watermark
df_watermarked = df_stream.withWatermark("event_time", "1 minutes")

# Group by window + name
agg_with_watermark = (
    df_watermarked
        .groupBy(window("event_time", "2 minutes"), "name")
        .count()        
)

agg_with_watermark_query = (
    agg_with_watermark.writeStream
                .format("delta")                       # ✅ Delta table sink
                .option("checkpointLocation", checkpointPath + "/with_watermark")
                .outputMode("complete")                # required for aggregations
                .option("mergeSchema", "true")
                .trigger(processingTime="5 seconds")  # micro-batch interval
                .toTable("agg_with_watermark_query")
)

# ------------------------------------------------------------
# 8) Wait for Completion
# ------------------------------------------------------------
agg_with_watermark_query.awaitTermination()

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 29, Finished, Cancelled, Cancelled)

In [30]:
df = spark.sql("SELECT * FROM test_Lakehouse.agg_with_watermark_query")
df.show(truncate=False)

StatementMeta(, ae0a0b5f-c154-4dda-b124-3de9614fc429, 32, Finished, Available, Finished)

+------------------------------------------+------+-----+
|window                                    |name  |count|
+------------------------------------------+------+-----+
|{2025-08-18 10:00:00, 2025-08-18 10:02:00}|Ayesha|1    |
|{2025-08-18 09:58:00, 2025-08-18 10:00:00}|Fatima|1    |
|{2025-08-18 10:00:00, 2025-08-18 10:02:00}|Ahmed |1    |
|{2025-08-18 09:56:00, 2025-08-18 09:58:00}|Usman |1    |
|{2025-08-18 09:58:00, 2025-08-18 10:00:00}|Bilal |1    |
|{2025-08-18 09:56:00, 2025-08-18 09:58:00}|Zara  |1    |
|{2025-08-18 10:00:00, 2025-08-18 10:02:00}|Ali   |1    |
+------------------------------------------+------+-----+

